### 벡터 임베딩 모델 nlpai-lab/KURE-v1 불러오기

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('nlpai-lab/KURE-v1')

v = model.encode("철골공사 중 추락 위험")
print(v.shape)     # (1024,) 나오면 성공

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

(1024,)


In [3]:
import json, numpy as np, torch
from sentence_transformers import SentenceTransformer

# sif.jsonl 읽기
with open('sif.jsonl', encoding='utf-8') as f:
    recs = [json.loads(line) for line in f]

texts = [r['검색문장'] for r in recs]         
print(f'{len(recs):,}건 로드')
print('예시:', texts[0][:60], '...')

model = SentenceTransformer('nlpai-lab/KURE-v1', device='cuda')
model.max_seq_length = 512                    
print('GPU:', torch.cuda.is_available(), '| 차원:', model.get_sentence_embedding_dimension())



3,459건 로드
예시: 토공사 굴착 작업 굴착 장비반입 중 바닥개구부(자재인양구 등)(으)로 인한 추락 — 굴착기 진입을 위해 현장 ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

GPU: True | 차원: 1024


C:\Users\songb\AppData\Local\Temp\ipykernel_29352\2152969190.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('GPU:', torch.cuda.is_available(), '| 차원:', model.get_sentence_embedding_dimension())


In [4]:
vecs = model.encode(
    texts,
    batch_size=16,              
    normalize_embeddings=True,    
    show_progress_bar=True,
    convert_to_numpy=True,
).astype('float32')

print(vecs.shape)  


Batches:   0%|          | 0/217 [00:00<?, ?it/s]

(3459, 1024)


In [5]:
np.save('sif.npy', vecs)

# 검증
chk = np.load('sif.npy')
print('shape   :', chk.shape, '| dtype:', chk.dtype)
print('용량    : %.1f MB' % (chk.nbytes/1024/1024))
print('정규화  :', np.allclose(np.linalg.norm(chk, axis=1), 1.0))
print('레코드 수 일치:', len(chk) == len(recs))


shape   : (3459, 1024) | dtype: float32
용량    : 13.5 MB
정규화  : True
레코드 수 일치: True


In [6]:
def 검색(질문, k=5):
    q = model.encode(질문, normalize_embeddings=True).astype('float32')
    점수 = vecs @ q                              # 내적 = 코사인 유사도
    상위 = np.argsort(-점수)[:k]
    for 순위, i in enumerate(상위, 1):
        r = recs[i]
        print(f"[{순위}] {점수[i]:.3f}  {r['공종']} / {r['작업명']} / {r['재해종류']}")
        print(f"     기인물: {r['기인물']}")
        print(f"     대책: {r['대책'][0][:60]}...")
        print()

검색("철골 구조물 조립 중 고소작업 추락 위험")


[1] 0.768  철골공사 / 철골 작업 / 추락
     기인물: 고소작업대(차)
     대책: 고소작업대를 사용하는 경우 탑승자가 작업대를 벗어나지 않도록 조치...

[2] 0.767  철골공사 / 철골 작업 / 추락
     기인물: 고소작업대(차)
     대책: 추락위험이 있는 철골 보에서 안전대를 안전대 부착설비 또는 철골 홀 등에 체결한 상태에서 이동...

[3] 0.761  철골공사 / 철골 작업 / 추락
     기인물: 고소작업대(차)
     대책: 고소작업대를 사용하는 경우 탑승자가 작업대를 벗어나지 않도록 조치...

[4] 0.759  철골공사 / 철골 작업 / 추락
     기인물: 고소작업대(차)
     대책: 고소작업대를 사용하는 경우 탑승자가 작업대를 벗어나지 않도록 조치...

[5] 0.758  철골공사 / 철골 작업 / 추락
     기인물: 철골구조물
     대책: 철골보 상부 등 추락위험이 있는 장소에서 작업 시 작업발판 및 고소작업대 사용하여 작업...



### 비슷한 사례 filter 형식으로 추출

In [7]:
import json, re, numpy as np
from build_sif import 재해종류_MAP, 공종_MAP, 작업종류_MAP

recs = [json.loads(l) for l in open('sif.jsonl', encoding='utf-8')]
vecs = np.load('sif.npy')

공종목록 = np.array([r['공종']   for r in recs])
작업목록 = np.array([r['작업명'] for r in recs])
재해목록 = np.array([r['재해종류'] for r in recs])


def 후보찾기(공종=None, 작업종류=None, 예측유형=None, 최소=10):
    """예측 결과 → 번역표 → 후보 인덱스. 후보가 적으면 조건을 단계적으로 푼다."""
    t공, t작, t재 = 공종_MAP.get(공종), 작업종류_MAP.get(작업종류), 재해종류_MAP.get(예측유형)
    m공 = (공종목록 == t공)        if t공 else None
    m작 = np.isin(작업목록, t작)   if t작 else None
    m재 = np.isin(재해목록, t재)   if t재 else None

    def 합(*ms):
        ms = [m for m in ms if m is not None]
        if not ms: return np.ones(len(recs), bool)
        out = ms[0].copy()
        for m in ms[1:]: out &= m
        return out

    for 이름, mask in [('공종+작업+재해', 합(m공, m작, m재)),
                       ('공종+재해',      합(m공, m재)),
                       ('공종+작업',      합(m공, m작)),
                       ('재해만',         합(m재)),
                       ('전체',           합())]:
        idx = np.flatnonzero(mask)
        if len(idx) >= 최소:
            return idx, 이름
    return np.arange(len(recs)), '전체'


In [8]:
_공백 = re.compile(r'[\s·,()]+')
def _키(대책):                       # 중복 판정용: 공백·기호 제거 후 앞 35자
    return _공백.sub('', 대책)[:35]


def 추천(공종, 작업종류, 예측유형, 상황, k=5, 풀=40):
    후보, 단계 = 후보찾기(공종, 작업종류, 예측유형)
    q = model.encode(상황, normalize_embeddings=True).astype('float32')
    점수 = vecs[후보] @ q

    본, 결과 = set(), []
    for j in np.argsort(-점수)[:풀]:
        i, s = 후보[j], float(점수[j])
        for 대책 in recs[i]['대책']:
            kk = _키(대책)
            if kk in 본: continue
            본.add(kk)
            결과.append({'대책': 대책, '점수': s, '출처': recs[i]})
            if len(결과) >= k:
                return 결과, 단계, len(후보)
    return 결과, 단계, len(후보)


In [9]:
for 공종, 작업, 유형, 상황 in [
    ("철근콘크리트공사", "콘크리트 타설", "추락·압착", "슬래브 콘크리트 타설 중 거푸집 붕괴 위험"),
    ("터널공사",        "굴착/토공사",   "물체에 맞음", "터널 막장 굴착 중 낙반 위험"),
    ("마감공사",        "마감 작업",     "추락·압착", "외벽 판넬 설치 중 달비계 작업"),
]:
    결과, 단계, n = 추천(공종, 작업, 유형, 상황)
    print(f'\n▶ {공종} / {작업} / {유형}   [{단계}, 후보 {n}건]')
    for 순위, r in enumerate(결과, 1):
        o = r['출처']
        print(f'  {순위}. ({r["점수"]:.3f}) {r["대책"][:70]}')
        print(f'        ↳ #{o["id"]} {o["공종"]}/{o["작업명"]}/{o["재해종류"]}')



▶ 철근콘크리트공사 / 콘크리트 타설 / 추락·압착   [공종+작업+재해, 후보 68건]
  1. (0.773) 거푸집 조립 시 구조검토한 후 조립도 작성 및 준수
        ↳ #576 철근콘크리트 공사/콘크리트 작업/붕괴
  2. (0.773) 콘크리트 타설시 집중타설 금지 및 작업시 거푸집 변형, 변위 등 점검 후 보수
        ↳ #576 철근콘크리트 공사/콘크리트 작업/붕괴
  3. (0.773) 데크플레이트의 양단 걸침길이 확보 및 고정철저
        ↳ #576 철근콘크리트 공사/콘크리트 작업/붕괴
  4. (0.767) 데크플레이트 받침 설치시 자중 및 작업하중을 견딜 수 있는 받침 설치
        ↳ #574 철근콘크리트 공사/콘크리트 작업/붕괴
  5. (0.767) 받침 설치 방법에 대한 시공상세도 작성 및 준수
        ↳ #574 철근콘크리트 공사/콘크리트 작업/붕괴

▶ 터널공사 / 굴착/토공사 / 물체에 맞음   [공종+작업, 후보 17건]
  1. (0.724) 터널 지보공 설치작업 시 숏크리트 타설방법 및 두께 관리
        ↳ #2401 터널공사/터널 굴착 작업/붕괴
  2. (0.724) 록볼트 정착재의 적절한 충진여부 확인
        ↳ #2401 터널공사/터널 굴착 작업/붕괴
  3. (0.724) 기계굴착 및 천공작업으로 막장면에 충격이 가해질 경우 막장면 숏크리트 등의 보강대책 적용
        ↳ #2401 터널공사/터널 굴착 작업/붕괴
  4. (0.724) 터널 굴착작업 시 주변 지질 및 지층의 상태 사전조사 실시
        ↳ #2401 터널공사/터널 굴착 작업/붕괴
  5. (0.682) 낙반부위의 절리상태 및 낙반위치 확인하여 떨어질 암석의 상태 파악
        ↳ #2394 터널공사/터널 굴착 작업/낙하

▶ 마감공사 / 마감 작업 / 추락·압착   [공종+작업+재해, 후보 850건]
  1. (0.733) 고소 작업 시 현장 내 안전을 고려하여 가능한 가장 안전한 

### LLM을 통해 자연스러운 언어로 사용자에게 보여주기

- 모델: **`LGAI-EXAONE/EXAONE-4.0-1.2B`** (bf16 2.56GB) — 측정으로 채택
- LLM은 검색된 KOSHA 원문 대책을 **옮겨 적기만** 한다. 새 대책을 만들지 않는다.

| 시험 구성 | 결과 |
|---|---|
| kanana-2-3b (4bit) | ❌ 출력 붕괴 — 자리표시자 복사, 같은 줄 30회 반복, 코드 조각 출력 |
| kanana-2-3b + RoPE factor 8.0 보정 | ❌ 회복 안 됨 (항목 0/5) |
| **EXAONE-4.0-1.2B + 개선 프롬프트** | ✅ **항목 5/5 × 3시나리오, 지어낸 수치 0건** |

3B가 1.2B보다 낫다는 보장은 없다. 파라미터 수가 아니라 측정 결과로 고른다.


In [14]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM

model.to('cpu')                       # KURE-v1을 CPU로 (질문 1개 인코딩은 CPU로 충분)
gc.collect(); torch.cuda.empty_cache()
print('GPU 비움: %.2f GB' % (torch.cuda.memory_allocated()/1e9))

# EXAONE-4.0-1.2B (bf16 2.56GB). 양자화 불필요 → bitsandbytes/accelerate 안 씀.
#
# ※ kanana-2-3b(4bit)를 시험했으나 출력이 붕괴하여 철회했다.
#   config 의 YaRN 설정이 어긋나 있다 — max_position 32768 / original 4096 이면
#   factor 가 8.0 이어야 하는데 40.0 으로 박혀 있다 (5배). transformers 가 경고를 띄운다.
#   factor 를 8.0 으로 보정해 재시험해도 코드 조각·빈 출력이 나와 회복되지 않았다.
#   파라미터가 크다고 결과가 좋은 것은 아니다 — 측정해서 나은 쪽을 쓴다.
LLM_ID = 'LGAI-EXAONE/EXAONE-4.0-1.2B'

tok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=torch.bfloat16).to('cuda')
llm.eval()
print('LLM 로드 완료: VRAM %.2f GB' % (torch.cuda.memory_allocated()/1e9))


GPU 비움: 2.14 GB


Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

LLM 로드 완료: VRAM 4.04 GB


In [15]:
# ★ 자리표시자 대신 "실제로 채워진 예시"를 준다.
#   "(대책)" 같은 괄호 자리표시자를 주면 모델이 그걸 글자 그대로 베껴 쓴다.
SYSTEM = """너는 건설현장 안전관리자를 돕는 assistant다.
[참고 사례]로 주어진 대책을 현장용 문장으로 옮겨 적는 것이 네 임무다.

규칙:
- 참고 사례에 없는 내용을 추가하지 마라. 숫자·횟수·치수를 새로 만들지 마라.
- 원인 분석이나 추측을 쓰지 마라.
- 참고 사례를 빠뜨리지 말고 전부 항목으로 만들어라.
- 원문 표현을 유지하고 어미만 "~하십시오"로 바꿔라.
- 같은 문장을 반복하지 마라. 표, 코드, 머리말을 쓰지 마라.
- 아래 예시와 똑같은 짜임으로 쓰고, 마지막 항목을 쓴 즉시 멈춰라.

── 예시 (형식만 참고. 내용은 절대 가져다 쓰지 마라) ──
■ 핵심 위험
철골공사 철골 작업 중 추락·압착 사고 발생 시 치명 위험 상위 12%입니다.

■ 안전 조치사항
1. 추락위험이 있는 장소에서 작업 시 작업발판 및 고소작업대를 사용하십시오. (사례 #770)
2. 철골구조물 승하강 시 안전대 부착설비나 안전블럭을 활용하십시오. (사례 #802)
──────────────────────────"""


def _정리(t: str) -> str:
    """반복된 줄과 형식 군더더기를 잘라낸다 (greedy 디코딩의 반복 루프 방어)."""
    줄, 본 = [], set()
    for l in t.splitlines():
        s = l.strip()
        if not s:
            줄.append('')
            continue
        if s in 본:                       # 똑같은 줄이 또 나오면 버림
            continue
        본.add(s)
        줄.append(l.rstrip())
    t = '\n'.join(줄)
    t = re.sub(r'^\s*[─—-]{3,}.*$', '', t, flags=re.M)               # 구분선
    t = re.sub(r'\n\s*(출력 완료\.?|이상입니다\.?|\(.*개수만큼\))\s*', '\n', t)
    return re.sub(r'\n{3,}', '\n\n', t).strip()


def 생성(현장, 예측, 결과, max_new_tokens=640):
    # 번호를 매겨서 주면 모델이 개수를 맞춰 출력한다 (유사도는 제외 — 지어낸 수치와 섞임)
    사례 = "\n".join(
        f"{n}. {r['대책']}  [사례 #{r['출처']['id']}]"
        for n, r in enumerate(결과, 1)
    )
    user = f"""[현장 조건]
{현장}

[예측 결과]
사고 발생 시 예상 유형: {예측['injury_top']}
치명 위험도: 상위 {100 - 예측['relative_risk_percentile']:.0f}%

[참고 사례]  — 아래 {len(결과)}개를 전부 사용하라
{사례}"""

    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user",   "content": user}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda')
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           do_sample=False,               # 안전 정보 → 무작위성 제거
                           repetition_penalty=1.12,       # ★ 같은 줄 반복 억제
                           no_repeat_ngram_size=18,
                           pad_token_id=tok.eos_token_id)
    답 = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return _정리(답)


In [16]:
# ── 생성 결과 자동 검증 ─────────────────────────────────────
# 작은 모델은 (1) 원문에 없는 수치를 끼워 넣거나 (2) 문장 순서를 뒤집는다.
# 안전 지침에서는 둘 다 치명적이므로 생성할 때마다 기계적으로 걸러낸다.
_숫자 = re.compile(r'\d+(?:\.\d+)?\s*(?:회|번|개|일|주|개월|시간|분|m|cm|mm|kg|톤|%|배|명|층)')


def _겹침(a, b):
    """두 문장의 2글자 조각이 얼마나 겹치는지 (0~1). 과도한 재작성 탐지용."""
    A = {a[i:i+2] for i in range(len(a)-1)}
    B = {b[i:i+2] for i in range(len(b)-1)}
    return len(A & B) / max(len(A), 1)


def 검증(답, 결과, 예측):
    허용 = {f"{100 - 예측['relative_risk_percentile']:.0f}%"}
    for r in 결과:
        허용 |= set(_숫자.findall(r['대책']))
    본문 = re.sub(r'사례\s*#\d+', '', 답)          # 출처 표기는 검사 대상에서 제외
    지어낸수치 = [t for t in _숫자.findall(본문) if t not in 허용]
    가짜출처 = set(re.findall(r'#(\d+)', 답)) - {str(r['출처']['id']) for r in 결과}

    # 출력 항목별로, 가장 가까운 원문 대책과 얼마나 겹치는지 본다.
    항목 = re.findall(r'^\s*\d+\.\s*(.+)$', 답, re.M)
    과다재작성 = []
    for 줄 in 항목:
        본 = _공백.sub('', re.sub(r'[—\-]?\s*\[?사례\s*#\d+\]?', '', 줄))
        점수 = max((_겹침(본, _공백.sub('', r['대책'])) for r in 결과), default=0)
        if 점수 < 0.25:            # 실측 기준. 충실한 재작성 0.3~0.4, 의미 변형 0.2 이하
            과다재작성.append(f'{점수:.2f} | {줄[:45]}')

    return {'지어낸수치': 지어낸수치, '가짜출처': sorted(가짜출처),
            '사용대책': f'{len(항목)}/{len(결과)}', '과다재작성': 과다재작성}


def 안전수칙(공종, 작업종류, 예측유형, 상황, 예측, k=5):
    결과, 단계, n후보 = 추천(공종, 작업종류, 예측유형, 상황, k=k)
    답변 = 생성(f"{공종} / {작업종류} — {상황}", 예측, 결과)
    return {'답변': 답변, '근거': 결과, '필터단계': 단계, '후보수': n후보,
            '검증': 검증(답변, 결과, 예측)}


### 시나리오 테스트

In [17]:
시나리오 = [
    ("철근콘크리트공사", "콘크리트 타설", "추락·압착", "슬래브 콘크리트 타설 중 거푸집 붕괴 위험",
     {'injury_top': '추락·압착', 'relative_risk_percentile': 82.0, 'risk_grade': '높음'}),
    ("마감공사", "마감 작업", "추락·압착", "외벽 판넬 설치 중 달비계 작업",
     {'injury_top': '추락·압착', 'relative_risk_percentile': 64.0, 'risk_grade': '보통'}),
    ("터널공사", "굴착/토공사", "물체에 맞음", "터널 막장 굴착 중 낙반 위험",
     {'injury_top': '물체에 맞음', 'relative_risk_percentile': 91.0, 'risk_grade': '매우 높음'}),
]

for 공종, 작업, 유형, 상황, 예측 in 시나리오:
    out = 안전수칙(공종, 작업, 유형, 상황, 예측)
    v = out['검증']
    print('=' * 74)
    print(f"{공종} / {작업} / {유형}   [{out['필터단계']}, 후보 {out['후보수']}건]")
    print('=' * 74)
    print(out['답변'])
    print(f"\n[검증] 지어낸 수치: {v['지어낸수치'] or '없음 OK'}"
          f" | 없는 출처: {v['가짜출처'] or '없음 OK'}"
          f" | 대책 사용: {v['사용대책']}")
    if v['과다재작성']:
        print('  ⚠ 원문과 많이 달라진 항목 — 순서·의미 뒤집힘 확인 필요')
        for x in v['과다재작성']:
            print('    ', x)
    print('[근거 원문]')
    for r in out['근거']:
        print(f"   #{r['출처']['id']} ({r['점수']:.2f}) {r['대책'][:60]}")
    print()

print('최대 VRAM: %.2f GB' % (torch.cuda.max_memory_allocated()/1e9))


철근콘크리트공사 / 콘크리트 타설 / 추락·압착   [공종+작업+재해, 후보 68건]
■ 핵심 위험
슬래브 콘크리트 타설 중 거푸집 붕괴로 인한 추락·압착 사고 발생 가능성 상위 18%입니다.

■ 안전 조치사항
1. 거푸집 조립 전 반드시 구조검토를 실시하고 조립도를 작성하여 준수하십시오. [사례 #576]
2. 타설 시 집중적인 하중 집중을 방지하기 위해 간격을 두고 순차적으로 타설하십시오. [사례 #576]
3. 데크플레이트 양단 고정을 확인하고 충분한 걸침길이 확보를 점검 후 보수하십시오. [사례 #576]
4. 받침 구조물이 자중 및 작업하중을 견디도록 설계 기준에 맞게 설치하십시오. [사례 #574]
5. 받침 설치 공정별 상세도를 명시한 도면을 작성하여 모든 작업자가 준수하도록 하십시오. [사례 #574]

[검증] 지어낸 수치: 없음 OK | 없는 출처: 없음 OK | 대책 사용: 5/5
  ⚠ 원문과 많이 달라진 항목 — 순서·의미 뒤집힘 확인 필요
     0.35 | 거푸집 조립 전 반드시 구조검토를 실시하고 조립도를 작성하여 준수하십시오. [사례
     0.12 | 타설 시 집중적인 하중 집중을 방지하기 위해 간격을 두고 순차적으로 타설하십시오.
     0.35 | 데크플레이트 양단 고정을 확인하고 충분한 걸침길이 확보를 점검 후 보수하십시오. 
     0.32 | 받침 구조물이 자중 및 작업하중을 견디도록 설계 기준에 맞게 설치하십시오. [사례
     0.19 | 받침 설치 공정별 상세도를 명시한 도면을 작성하여 모든 작업자가 준수하도록 하십시
[근거 원문]
   #576 (0.77) 거푸집 조립 시 구조검토한 후 조립도 작성 및 준수
   #576 (0.77) 콘크리트 타설시 집중타설 금지 및 작업시 거푸집 변형, 변위 등 점검 후 보수
   #576 (0.77) 데크플레이트의 양단 걸침길이 확보 및 고정철저
   #574 (0.77) 데크플레이트 받침 설치시 자중 및 작업하중을 견딜 수 있는 받침 설치
   #574 (0.77